<a href="https://colab.research.google.com/github/AmirMohammad73/semantic_folding/blob/main/tsne_fingerprint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE  # Import t-SNE

# Set a seed for reproducibility
np.random.seed(123)
DIMENSION = 16

# Load corpus
with open("/content/drive/MyDrive/cleaned_full_openbook.txt", "r") as file:
    corpus = file.readlines()

# TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=4146)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

# Dimensionality reduction using TruncatedSVD
n_components = 4146
svd = TruncatedSVD(n_components=n_components)
document_vectors = svd.fit_transform(tfidf_matrix)

# Clustering with t-SNE (setting the random seed)
tsne = TSNE(n_components=2, perplexity=10, random_state=123)
document_vectors_tsne = tsne.fit_transform(document_vectors)

# Step 1: Get the range for the document_vectors_tsne
x_min, x_max = document_vectors_tsne[:, 0].min(), document_vectors_tsne[:, 0].max()
y_min, y_max = document_vectors_tsne[:, 1].min(), document_vectors_tsne[:, 1].max()

# Step 2: Create a 16x16 grid to bin the data
num_bins = DIMENSION
x_edges = np.linspace(x_min, x_max, num_bins + 1)
y_edges = np.linspace(y_min, y_max, num_bins + 1)

# Step 3: Adjusted calculation of bin indices to ensure they fall within 0-15
# Avoid the last edge to ensure the value falls within the 0-15 range
x_bin_indices = np.clip(np.digitize(document_vectors_tsne[:, 0], bins=x_edges[:-1]) - 1, 0, DIMENSION - 1)
y_bin_indices = np.clip(np.digitize(document_vectors_tsne[:, 1], bins=y_edges[:-1]) - 1, 0, DIMENSION - 1)

# Now, all bin indices should be within the range [0, DIMENSION - 1], which means 0 to 15

# Step 4: Make a list of 16x16 grid coordinates plus document IDs
mapped_data = list(zip(x_bin_indices, y_bin_indices, range(len(document_vectors_tsne))))

# Convert to a numpy array if necessary
mapped_data_array = np.array(mapped_data)

# Initialize a fourth column in 'mapped_data_array' for the frequency counts, setting them initially to zero
mapped_data_array = np.hstack((mapped_data_array, np.zeros((mapped_data_array.shape[0], 1), dtype=int)))

# Prompt the user for words to search for (multiple words separated by space)
search_words = input("Enter the words to find (separated by space): ").split()

# Initialize a frequency counter for each document with a dict comprehension
word_frequencies = {doc_id: 0 for doc_id in range(len(corpus))}

# Scan through the corpus to find the exact frequency of each search word in each document
for doc_id, document in enumerate(corpus):
    # Convert the document to lowercase and split into words
    words = document.lower().split()
    # Count the exact occurrences of each search_word in the words list
    for search_word in search_words:
        word_frequencies[doc_id] += words.count(search_word.lower())

# Update 'mapped_data_array' to include the total word count
for i, (x_index, y_index, doc_id) in enumerate(mapped_data):
    # Note that 'word_frequencies' is a dictionary where the keys are document IDs and the values are counts
    total_word_count = word_frequencies[doc_id]
    # Update the total word count in the fourth column
    mapped_data_array[i, 3] = total_word_count

# The resulting array now has the total word count in the fourth column
# Display the output
# Optionally, you can filter out the entries with a total word count of zero to only include documents where the words appear
filtered_mapped_data = mapped_data_array[mapped_data_array[:, 3] > 0]

# Initialize the 16x16 matrix with zeros
matrix = np.zeros((DIMENSION, DIMENSION), dtype=int)

# Iterate over the filtered_mapped_data to populate the matrix
for data in filtered_mapped_data:
    x_coord, y_coord, doc_id, total_word_count = data
    matrix[y_coord, x_coord] += total_word_count  # Note that y_coord is used for rows, x_coord for columns

# Now, matrix contains the total word count at each coordinate
# To display the 16x16 matrix, just print it
print(matrix)


Enter the words to find (separated by space): What is a source of energy
[[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  1  0  0  0  1  1  0  0  0  0  0  0  0]
 [ 0  0  2  0  0  2  0  1  1  0  0  0  0  0  0  0]
 [ 0  0  1  3  0  0  5  0  0  0  0  0  0  0  0  0]
 [ 0  5  0  0  3 12  1  0  0  0  1  0  0  0  0  0]
 [ 0  5  1  7  7  0  0  1  1  2  0  0  0  0  0  0]
 [ 0  5  1  2  2  0  0  0  0  0  2  2  1  0  0  0]
 [ 0  4  2  2  2  4  1  1  0  1  7  0  0  0  0  0]
 [ 1  1  4  3  4 31  0  1  3  3  0  0  0  0  0  0]
 [ 0  1  0  6  6  6 17  0  0  1  0  1  0  0  0  0]
 [ 0  1  2  8 14  0  1 10  1  2  0  1  2  0  1  0]
 [ 0  2 10 32 17  0  1  0  0  0  0  0  0  0  0  0]
 [ 0  0  1  3  0  0  1  1  0  0  3  0  0  0  0  0]
 [ 0  0  0  0  2  1  1  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]]


In [8]:
# Prompt the user for words to search for (multiple words separated by space)
search_words = input("Enter the words to find (separated by space): ").split()

# Initialize a frequency counter for each document with a dict comprehension
word_frequencies = {doc_id: 0 for doc_id in range(len(corpus))}

# Scan through the corpus to find the exact frequency of each search word in each document
for doc_id, document in enumerate(corpus):
    # Convert the document to lowercase and split into words
    words = document.lower().split()
    # Count the exact occurrences of each search_word in the words list
    for search_word in search_words:
        word_frequencies[doc_id] += words.count(search_word.lower())

# Update 'mapped_data_array' to include the total word count
for i, (x_index, y_index, doc_id) in enumerate(mapped_data):
    # Note that 'word_frequencies' is a dictionary where the keys are document IDs and the values are counts
    total_word_count = word_frequencies[doc_id]
    # Update the total word count in the fourth column
    mapped_data_array[i, 3] = total_word_count

# The resulting array now has the total word count in the fourth column
# Display the output
# Optionally, you can filter out the entries with a total word count of zero to only include documents where the words appear
filtered_mapped_data = mapped_data_array[mapped_data_array[:, 3] > 0]

# Initialize the 16x16 matrix with zeros
matrix = np.zeros((DIMENSION, DIMENSION), dtype=int)

# Iterate over the filtered_mapped_data to populate the matrix
for data in filtered_mapped_data:
    x_coord, y_coord, doc_id, total_word_count = data
    matrix[y_coord, x_coord] += total_word_count  # Note that y_coord is used for rows, x_coord for columns

# Now, matrix contains the total word count at each coordinate
# To display the 16x16 matrix, just print it
print(matrix)


Enter the words to find (separated by space): fuel
[[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  1  0  0  0  0  0  0  0  0  0  1  0  0  0]
 [ 0  0 21  1  2  0  0  0  0  0  0  0  1  0  0  0]
 [ 0  0  3  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]]
